In [1]:
import numpy as np
import random
import time
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit_aer import AerSimulator
import qiskit.qasm3
from qiskit_ibm_runtime.fake_provider import FakeVigoV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

In [2]:
measurements = 0
explosions = 0
detections = 0
brute_force = 100

In [3]:
def slicer(cells):
    if len(cells) >= 2 and len(cells) != 3:
        split_cells = [cells[:len(cells)//2],cells[len(cells)//2:]]
    elif len(cells) == 3:
        split_cells = [[a] for a in cells]
    else:
        split_cells = cells
    return list(reversed(split_cells))
    

In [4]:
def ifm_kwiat_scanner(bomb_sites,test_sites):
    cycles = 200 # Zeno Cycles
    qr = QuantumCircuit(101,18) # 1 Photon Qubit, 100 Path Qubits
    theta = np.pi/(2*cycles)    # Angle to rotate Photon Qubit about according to the amount of zeno cycles
    for i in range(cycles-1): # cycles-1 Zeno cycles done
        qr.rx(2*theta,0)    # Rotate Photon Qubit by Zeno Angle (Beamsplitter)
        for k in test_sites:
            qr.cx(0,k)  # Put path qubits that are to be tested into superposition into either not exploded or exploded
        qr.barrier()
        for b,bomb in enumerate(bomb_sites):
            qr.measure(bomb,b+1)    # Measures act as ships and test whether photon hit it or not
        qr.barrier()
        for k in test_sites:
            qr.cx(0,k)  # Undo ship superpositioning
        qr.barrier()
        for l in bomb_sites:
            qr.reset(l) # Reset ship qubits for next Zeno cycle
        qr.barrier()
    qr.rx(2*theta,0) # Final Zeno cycle completed as resets and bomb measures not needed (Final Beamsplitter for recombination)
    qr.measure(0,0) # Measure photon qubit, 0 is when ship is detected without interaction and 1 is when no ships in path or if ship is triggered
    result_qr = AerSimulator().run(qr,shots=1).result()
    qr_result = next(iter(result_qr.get_counts().keys()))
    # Keeping track of measurements made, if ships have been tripped and how many detections without interactions have been made
    global measurements
    global explosions
    global detections
    measurements += 1
    verdict = None
    if (qr_result[-1] == '0') and ('1' not in qr_result): # Full String of 0's mean ship was detected without interaction (path qubits not measured as 1)
        verdict = 'Detected'
        detections += 1
    elif (qr_result[-1] == '1') and ('1' not in qr_result[:-1]): # Photon qubit is 1 and path qubits are 0 so no ship detected and no interaction with ships, therefore no ships in path
        verdict = 'Empty'
    elif (qr_result[-1] == '1') and ('1' in qr_result[:-1]): # Photon qubit is 1 and some of the path qubits are 1 where photon has interacted with ship
        verdict = 'Exploded'
        explosions += 1
    return verdict

In [5]:
def simulator(bombs):
    ship_cells_no = 17
    grid = list(range(1,101))
    water = []
    ships = []
    regions = []
    for i in range(10):
        regions.append(grid[10*i:10*i + 10])
    i = 0
    while ship_cells_no > 0:
        print(regions)
        if len(regions) == 0:
            break
        else:
            verdict = ifm_kwiat_scanner([a for a in bombs if a in regions[0]],regions[0]) 
                
            
            if verdict == 'Empty':
                water.append(regions.pop(0)) # Remove non-ship segements from scanning space
                
            elif verdict == 'Detected':
                if len(regions[0]) == 2:
                    verdict_small = ifm_kwiat_scanner([a for a in regions[0] if a in bombs],[(regions[0])[0]])
                    if verdict_small == 'Empty': # If length 2 segment has ship in it and first cell is empty, other cell must contain ship
                        ships.append((regions[0])[1])
                        water.append((regions[0])[0])
                        regions.pop(0)
                        ship_cells_no -= 1
                    if verdict_small == 'Detected': # If length 2 segment has ship in it and first cell has ship, other cell could be ship as well
                        ships.append((regions[0])[0])
                        ship_cells_no -= 1
                        verdict_cell = ifm_kwiat_scanner([a for a in regions[0] if a in bombs],[(regions[0])[1]])
                        if verdict_cell == 'Empty':
                            water.append((regions[0])[1])
                            regions.pop(0)
                        if verdict_cell == 'Detected':
                            ships.append((regions[0])[1])
                            regions.pop(0)
                            ship_cells_no -= 1
                elif len(regions[0]) == 1:
                    verdict_small = ifm_kwiat_scanner([a for a in bombs if a in regions[0]],regions[0])
                    if verdict_small == 'Detected':
                        ships.append((regions.pop(0))[0])
                        ship_cells_no -= 1
                    elif verdict_small == 'Empty':
                        water.append((regions.pop(0))[0])
                

                else: 
                    # Recursively split segments into smaller pieces to hone in on ship cells
                    sliced_cells = slicer(regions[0])
                    regions.pop(0) 
                    if len(sliced_cells) == 1:
                        regions.insert(0,sliced_cells)
                    else:
                        for i in sliced_cells:
                            regions.insert(0,i)
                                  
    return ships



In [6]:
def generate_random_grid():
    ships = {"Carrier": 5,"Battleship": 4,"Cruiser": 3,"Submarine": 3,"Destroyer": 2}
    random.seed()
    grid = np.zeros((10, 10), dtype=int)
    for ship_length in ships.values():
        ship_placed = False
        while not ship_placed:
            orientation = random.choice(["Horizontal", "Vertical"])
            if orientation == "Horizontal":
                row = random.randint(0, 9)
                col = random.randint(0, 10 - ship_length)
                coords = [(row, col + i) for i in range(ship_length)]
            else:
                row = random.randint(0, 10 - ship_length)
                col = random.randint(0, 9)
                coords = [(row + i, col) for i in range(ship_length)]
            if all(grid[i, j] == 0 for i, j in coords):
                for i, j in coords:
                    grid[i, j] = 1
                ship_placed = True

    return grid.ravel()


In [7]:
def print_grid(bombs):
    alphabet = 'ABCDEFGHIJ'
    print('  1 2 3 4 5 6 7 8 9 10')
    print(" ┌" + "--" * 9 + "-┐")
    for i in range(10):
        print(f'{alphabet[i]}|{int(bombs[10*i])} {int(bombs[10*i + 1])} {int(bombs[10*i+2])} {int(bombs[10*i+3])} {int(bombs[10*i+4])} {int(bombs[10*i+5])} {int(bombs[10*i+6])} {int(bombs[10*i+7])} {int(bombs[10*i+8])} {int(bombs[10*i+9])}|')
    print(" └" + "––" * 9 + "-┘\n")
    

In [8]:
def generate_user_grid():
    alphabet = 'ABCDEFGHIJ'
    print('  1 2 3 4 5 6 7 8 9 10')
    print(" ┌" + "--" * 9 + "-┐")
    for i in range(10):
        print(f'{alphabet[i]}|? ? ? ? ? ? ? ? ? ?|')
    print(" └" + "––" * 9 + "-┘\n")
    ship_coords_array = np.zeros((10,10))
    ships = {'Carrier': '1', 'Battleship': '2', 'Cruiser' : '3', 'Submarine' : '4', 'Destroyer': '5'}
    while len(ships) > 0:
        print('~ Which ship would you like to place? ~')
        for key,value in ships.items():
            print(f'{value}. {key}')
        place_ship = input('~ Enter the ship number with no \'.\': ~\n')
        if place_ship not in ships.values():
            place_ship = input('~ Enter a valid ship: ~\n')
        print('~ Which orientation would you like the ship? ~')
        print('~ 1. Vertical ~')
        print('~ 2. Horizontal ~')
        orientation = input('~ Enter orientation: 1 or 2 ~\n')
        if orientation not in ("1", "2"):
            print("~ Invalid orientation. ~\n")
            continue
        print('~ What coordinates would you like to place the end of the ship? Answer should be in form eg. (A5) ~')
        coordinate = input('~ Enter coordinate: ~')
        coord = coordinate.strip().upper()
        letter_index = alphabet.index(coord[0])
        number_index = int(coord[1:])-1
        if orientation == '1':
            if place_ship == '1':
                if letter_index + 5 > 10:
                    print("~ Try a coordinate that fits ~")
                    continue
                ship_coords = [(letter_index + i, number_index) for i in range(5)]
            if place_ship == '2':
                if letter_index + 4 > 10:
                    print("~ Try a coordinate that fits ~")
                    continue
                ship_coords = [(letter_index + i, number_index) for i in range(4)]
            if place_ship == '3':
                if letter_index + 3 > 10:
                    print("~ Try a coordinate that fits ~")
                    continue
                ship_coords = [(letter_index + i, number_index) for i in range(3)]
            if place_ship == '4':
                if letter_index + 3 > 10:
                    print("~ Try a coordinate that fits ~")
                    continue
                ship_coords = [(letter_index + i, number_index) for i in range(3)]
            if place_ship == '5':
                if letter_index + 2 > 10:
                    print("~ Try a coordinate that fits ~")
                    continue
                ship_coords = [(letter_index + i, number_index) for i in range(2)]
            

        elif orientation == '2':
            if place_ship == '1':
                if number_index + 5 > 10:
                    print("~ Try a coordinate that fits ~")
                    continue
                ship_coords = [(letter_index, number_index+i) for i in range(5)]
            if place_ship == '2':
                if number_index + 4 > 10:
                    print("~ Try a coordinate that fits ~")
                    continue
                ship_coords = [(letter_index, number_index+i) for i in range(4)]
            if place_ship == '3':
                if number_index + 3 > 10:
                    print("~ Try a coordinate that fits ~")
                    continue
                ship_coords = [(letter_index, number_index+i) for i in range(3)]
            if place_ship == '4':
                if number_index + 3 > 10:
                    print("~ Try a coordinate that fits ~")
                    continue
                ship_coords = [(letter_index, number_index+i) for i in range(3)]
            if place_ship == '5':
                if number_index + 2 > 10:
                    print("~ Try a coordinate that fits ~")
                    continue
                ship_coords = [(letter_index, number_index+i) for i in range(2)]
            
        if any(ship_coords_array[i,j] != 0 for i,j in ship_coords):
            print("~ Try a coordinate that fits ~")
            continue
        for i, j in ship_coords:
            ship_coords_array[i,j] = 1
  
        ships = {key:val for key, val in ships.items() if val != place_ship}
        
        print_grid(ship_coords_array.ravel())
    
    return ship_coords_array.ravel()

In [9]:
print('~ Welcome to Quantum Battleship ~\n')
generate_type = input('~ Please select if you would like to generate a random battleship grid (Enter: 1) or generate your own (Enter: 2) ~')
if generate_type == '1':
    grid_coords = generate_random_grid()
    ships = list(np.nonzero(grid_coords)[0]+1)
elif generate_type == '2':
    grid_coords = generate_user_grid()
    ships = list(np.nonzero(grid_coords)[0]+1)
else:
    input('~ Please enter a valid number ~')

detected_ships = simulator(ships)
detected_ships_coords = np.zeros(100)
for i in detected_ships:
    detected_ships_coords[i-1] = 1
print('\nGenerated Grid\n')
print_grid(grid_coords)
print('\n Detected Grid\n')
print_grid(detected_ships_coords)
print(f'\nMeasurements Made: {measurements}')
print(f'Detections: {detections}')
print(f'Ships Tripped: {explosions}')
if explosions > 0:
    print(f'\nScore: {(detections/explosions) * (measurements/brute_force)}')
if explosions == 0:
    print(f'\nScore: {(detections) * (measurements/brute_force)}')


~ Welcome to Quantum Battleship ~

[[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [11, 12, 13, 14, 15, 16, 17, 18, 19, 20], [21, 22, 23, 24, 25, 26, 27, 28, 29, 30], [31, 32, 33, 34, 35, 36, 37, 38, 39, 40], [41, 42, 43, 44, 45, 46, 47, 48, 49, 50], [51, 52, 53, 54, 55, 56, 57, 58, 59, 60], [61, 62, 63, 64, 65, 66, 67, 68, 69, 70], [71, 72, 73, 74, 75, 76, 77, 78, 79, 80], [81, 82, 83, 84, 85, 86, 87, 88, 89, 90], [91, 92, 93, 94, 95, 96, 97, 98, 99, 100]]
[[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15, 16, 17, 18, 19, 20], [21, 22, 23, 24, 25, 26, 27, 28, 29, 30], [31, 32, 33, 34, 35, 36, 37, 38, 39, 40], [41, 42, 43, 44, 45, 46, 47, 48, 49, 50], [51, 52, 53, 54, 55, 56, 57, 58, 59, 60], [61, 62, 63, 64, 65, 66, 67, 68, 69, 70], [71, 72, 73, 74, 75, 76, 77, 78, 79, 80], [81, 82, 83, 84, 85, 86, 87, 88, 89, 90], [91, 92, 93, 94, 95, 96, 97, 98, 99, 100]]
[[1, 2], [3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15, 16, 17, 18, 19, 20], [21, 22, 23, 24, 25, 26, 27, 28, 29, 30], [31, 32, 33, 34